In [1]:
from qiskit import __version__
print(__version__)

2.1.1


In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, state_fidelity, Pauli, DensityMatrix, partial_trace
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_aer.library import SaveDensityMatrix
from qiskit import transpile 
import numpy as np
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.circuit.library import HGate, UnitaryGate, IGate
import matplotlib.pyplot as plt

In [3]:
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))
amp_0 = np.cos(theta/2)
amp_1 = np.sin(theta/2)

# Infidelity to Error Rates

In [4]:
def inf_to_error(infidelity_list, num_qubits):
    error_rates_list = []
    dimension = 2**num_qubits
    for infidelity in infidelity_list:
        error_rates_list.append( infidelity * (dimension / (dimension - 1)) )
    
    return error_rates_list

In [5]:
one_qubit_gate_infidelity = [2.8e-5 * i for i in range(1,6)]
two_qubit_gate_infidelity = [8.3e-4 * i for i in range(1,6)]
idle_gate_infidelity = [1.2e-4 * i for i in range(1,6)]
spam0 = [6.7e-4 * i for i in range(1,6)] # P(1|0), measured 1 given that 0 was prepared
spam1 = [1.2e-3 * i for i in range(1,6)] # P(0|1)

In [6]:
one_qubit_gate_error_prob = inf_to_error(one_qubit_gate_infidelity, num_qubits=1)
two_qubit_gate_error_prob = inf_to_error(two_qubit_gate_infidelity, num_qubits=2)
idle_gate_error_prob = inf_to_error(idle_gate_infidelity, num_qubits=1)

# Non-FT Functions

Qubits are ordered from top to down, left to right, increasingly

The logical operators act on the left side of the triangle

In [29]:
def encoding(qc: QuantumCircuit, start: int):
    # This version allows for arbitrary state encoding by setting the first qubit to the arbitrary state
    for i in [2,3,5]:
        qc.h(start+i)
    
    cx_list = [[0,4], [0,1], [2,6], [2,4], [2,0], [5,6], [5,1], [5,0], [3,6], [3,1], [3,4]]
    for cx in cx_list:
        cx[0] += start 
        cx[1] += start 
        qc.cx(cx[0], cx[1])
    """
    # This version doesn't allow for arbitrary state encoding
    for i in [0,4,6]:
        qc.h(start+i)
        
    cx_list = [[0,1], [6,3], [0,3], [4,5], [4,2], [6,5], [4,1], [3,2]]
    for cx in cx_list:
        cx[0] += start
        cx[1] += start
        qc.cx(cx[0], cx[1])
    """

In [ ]:
def logical_exotic_magic_gate(qc: QuantumCircuit, start: int) -> bool:
    anc_qubits = QuantumRegister(7)
    anc_bits = ClassicalRegister(7)
    qc.add_register(anc_qubits, anc_bits)
    
    qc.ry(theta, anc_qubits[0])
    encoding(qc, start=anc_qubits[0])
    
    for i in range(7):
        qc.cy(qc.qubits[start + i], anc_qubits[i])
    
    qc.measure(anc_qubits[i for i in range(7)], anc_bits)
    

In [ ]:
def state_prep(qc: QuantumCircuit, start1: int, start2: int):
    encoding(qc, start1)
    for i in range(7):
        qc.h(i+start1)
    
    #s.conj().T @ hadamard @ s.conj().T @ ems @ ems @ s @ hadamard
    for i in range(7):
        qc.h(i+start2)
        qc.sdg(i+start2)
        

In [19]:
def syndrome_meas(qc, start, x_ancillas, z_ancillas, x_syndrome, z_syndrome):
    qc.h(x_ancillas)
    
    for i in [0,1,2,3]:
        qc.cx(start+i, z_ancillas[0])
        qc.cx(x_ancillas[0], start+i)
    for i in [1,2,4,5]:
        qc.cx(start+i, z_ancillas[1])
        qc.cx(x_ancillas[1], start+i)
    for i in [2,3,5,6]:
        qc.cx(start+i, z_ancillas[2])
        qc.cx(x_ancillas[2], start+i)
        
    qc.h(x_ancillas)
    
    qc.measure(x_ancillas, x_syndrome)
    qc.measure(z_ancillas, z_syndrome)

In [33]:
qc = QuantumCircuit(7)
qc.x(0)
encoding(qc, 0)
x_ancillas = QuantumRegister(3)
z_ancillas = QuantumRegister(3)
x_syndrome = ClassicalRegister(3)
z_syndrome = ClassicalRegister(3)

qc.add_register(x_ancillas, z_ancillas, x_syndrome, z_syndrome)
syndrome_meas(qc, 0, x_ancillas, z_ancillas, x_syndrome, z_syndrome)

In [34]:
noise_model = NoiseModel()
    
    
meas_circuit_noise = qc.copy()
#meas_circuit_noise.save_statevector(label='state_post', pershot=True, conditional=True)

backend = AerSimulator(noise_model=noise_model)
job = backend.run(meas_circuit_noise, shots=1)
result = job.result()

In [35]:
print(result)

Result(backend_name='aer_simulator', backend_version='0.17.1', job_id='7aa6e228-2c5e-493f-997a-e99376bf2a07', success=True, results=[ExperimentResult(shots=1, success=True, meas_level=2, data=ExperimentResultData(counts={'0x0': 1}), header={'creg_sizes': [['c6', 3], ['c7', 3]], 'global_phase': 0.0, 'memory_slots': 6, 'n_qubits': 13, 'name': 'circuit-174', 'qreg_sizes': [['q', 7], ['q6', 3], ['q7', 3]], 'metadata': {}}, status=DONE, seed_simulator=3704607259, metadata={'num_bind_params': 1, 'runtime_parameter_bind': False, 'parallel_state_update': 12, 'parallel_shots': 1, 'sample_measure_time': 1.7265e-05, 'noise': 'ideal', 'batched_shots_optimization': False, 'remapped_qubits': False, 'active_input_qubits': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'device': 'CPU', 'time_taken': 0.000220501, 'measure_sampling': True, 'num_clbits': 6, 'max_memory_mb': 32768, 'input_qubit_map': [[12, 12], [11, 11], [10, 10], [9, 9], [8, 8], [7, 7], [6, 6], [5, 5], [4, 4], [3, 3], [2, 2], [1, 1], [0, 0]